## Phase 2 — Transformer Encoder Autopsy

In this phase, we inspect the real pretrained ViT-Base model.

#### Data flow inside one encoder block

1. LayerNorm before attention
2. Query, Key, and Value projections
3. Multi-head self-attention
4. Attention output projection
5. First residual connection
6. LayerNorm before MLP
7. MLP expansion: 768 → 3072
8. GELU activation
9. MLP contraction: 3072 → 768
10. Second residual connection

The sequence shape remains `[B, 197, 768]` throughout the block.

In [1]:
import math
import torch
import torch.nn.functional as F
from transformers import AutoImageProcessor, ViTForImageClassification

import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

In [2]:
MODEL_ID = "google/vit-base-patch16-224"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

**Load Model**

In [3]:
model = ViTForImageClassification.from_pretrained(
    MODEL_ID,
    attn_implementation="eager"
).to(device)

model.eval()
print("Device:", device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Device: cuda


**Configuration X-ray**

In [4]:
config = model.config

print("Model type             :", config.model_type)
print("Image size             :", config.image_size)
print("Patch size             :", config.patch_size)
print("Hidden size            :", config.hidden_size)
print("Encoder blocks         :", config.num_hidden_layers)
print("Attention heads        :", config.num_attention_heads)
print("Intermediate/MLP size  :", config.intermediate_size)
print("Number of output labels:", config.num_labels)
print("Hidden activation      :", config.hidden_act)

Model type             : vit
Image size             : 224
Patch size             : 16
Hidden size            : 768
Encoder blocks         : 12
Attention heads        : 12
Intermediate/MLP size  : 3072
Number of output labels: 1000
Hidden activation      : gelu


In [5]:
head_dimension = (
    config.hidden_size // config.num_attention_heads
)

num_of_patches = (
    config.image_size // config.patch_size
) ** 2

total_tokens = num_of_patches + 1

print("Head dimension :", head_dimension)
print("Patch tokens   :", num_of_patches)
print("Total tokens   :", total_tokens)

Head dimension : 64
Patch tokens   : 196
Total tokens   : 197


**Parameter count**

In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

parameter_memory_mb = (
    total_parameters * 4 / (1024 ** 2)
)

print(f"Total parameters    : {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"FP32 parameter memory: {parameter_memory_mb:.2f} MB")

Total parameters    : 86,567,656
Trainable parameters: 86,567,656


#### Part B — Real patch projection inspect

**Patch projection layer**

In [7]:
patch_projection = (
    model.vit.embeddings.patch_embeddings.projection
)

print(patch_projection)
print("Weight shape:", patch_projection.weight.shape)
print("Bias shape:", patch_projection.bias.shape)

Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
Weight shape: torch.Size([768, 3, 16, 16])
Bias shape: torch.Size([768])


_CLS and position embedding_

In [12]:
cls_token = model.vit.embeddings.cls_token
position_embeddings = (
    model.vit.embeddings.position_embeddings
)

print("CLS token shape         :", cls_token.shape)
print("Position embedding shape:", position_embeddings.shape)

CLS token shape         : torch.Size([1, 1, 768])
Position embedding shape: torch.Size([1, 197, 768])


**Architecture shape trace**

In [ ]:
dummy_image = torch.zeros(
    1, config.num_channels,
    config.image_size, config.image_size,
    device=device
)

captured_patch_output = {}

def capture_path_projection(module, inputs, output):
    captured_patch_output["tensor"] = output.detach()

hook = patch_projection.register_forward_hook(
    capture_path_projection
)

In [ ]:
with torch.inference_mode():
    outputs = model(
        pixel_values=dummy_image,
        output_hidden_states=True,
        return_dict=True
    )
hook.remove()

In [ ]:
print("MODEL SHAPE TRACE")
print("=" * 45)

print("Input image          :",tuple(dummy_image.shape))
print("Conv patch output    :",tuple(captured_patch_output["tensor"].shape))
print("Number of states     :",len(outputs.hidden_states))
print("Embedding output     :",tuple(outputs.hidden_states[0].shape))

print("Block 1 output       :",tuple(outputs.hidden_states[1].shape))
print("Block 12 output      :",tuple(outputs.hidden_states[-1].shape))
print("Classification logits:",tuple(outputs.logits.shape))

In [ ]:
for index, hidden_state in enumerate(
    outputs.hidden_states
):
    if index == 0:
        layer_name = "Embedding"
    else:
        layer_name = f"Transformer block {index:02d}"

    print(
        f"{layer_name:<24}",
        tuple(hidden_state.shape)
    )

In [ ]:
batch_size = 1
tokens = total_tokens
heads = config.number_of_heads
head_dim = head_dimension

attention_shapes = {
    "Transformer input": (
        batch_size, tokens, config.hidden_size
    ),
    "Q before split": (
        batch_size, tokens, config.hidden_size
    ),
    "Q after split": (
        batch_size, heads, tokens, head_dim
    ),
    "Attention matrix": (
        batch_size, heads, tokens, tokens
    ),
    "Merged attention": (
        batch_size, tokens, config.hidden_size
    ),
    "MLP expansion": (
        batch_size, tokens, config.mlp_size
    ),
    "MLP contraction": (
        batch_size, tokens, config.hidden_size
    )
}

for name, shape in attention_shapes.items():
    print(f"{name:<24}: {shape}")